<a href="https://colab.research.google.com/github/Samlat-stack/spam-prediction/blob/main/spam_ham_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
import pandas as pd

url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep="\t", header=None, names=["label", "message"])
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [36]:
df.shape


(5572, 2)

In [37]:
df["label"].value_counts()

,count
label,
ham,4825
spam,747


In [38]:
df.isna().sum()

,0
label,0
message,0


In [39]:
# ham -> 0, spam -> 1 (same idea as male/female on Titanic)
df["label_num"] = df["label"].map({"ham": 0, "spam": 1})

# the dumb baseline: how often is "always guess ham" right?
print("Always-ham accuracy:", round((df["label_num"] == 0).mean(), 3))

Always-ham accuracy: 0.866


In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

vec = TfidfVectorizer()
X_train_vec = vec.fit_transform(X_train)   # learn the word list from training text, then convert
X_test_vec = vec.transform(X_test)         # convert test text using the same word list

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_vec, y_train)

print("Accuracy:", round(lr.score(X_test_vec, y_test), 3))
print(classification_report(y_test, lr.predict(X_test_vec), target_names=["ham", "spam"]))

Accuracy: 0.973
              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       966
        spam       1.00      0.80      0.89       149

    accuracy                           0.97      1115
   macro avg       0.98      0.90      0.94      1115
weighted avg       0.97      0.97      0.97      1115



In [41]:
from sklearn.pipeline import make_pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score

models = {
    "Logistic regression": make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000)),
    "Naive Bayes": make_pipeline(TfidfVectorizer(), MultinomialNB()),
}

for name, m in models.items():
    scores = cross_val_score(m, X, y, cv=5)
    print(name, "average:", round(scores.mean(), 3), "| spread:", round(scores.std(), 3))

Logistic regression average: 0.97 | spread: 0.003
Naive Bayes average: 0.958 | spread: 0.003
